In [ ]:
!pip install "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes

In [ ]:
%%writefile train.py
import torch
import os
from datasets import load_dataset
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template
from trl import SFTTrainer
from transformers import TrainingArguments, TrainerCallback

# Custom callback to handle checkpointing without triggering the pickling bug
class MultiGpuSaveCallback(TrainerCallback):
    def on_step_end(self, args, state, control, model=None, **kwargs):
        # Save every 500 steps, but only on Rank 0
        if state.global_step % 500 == 0 and state.global_step > 0:
            if args.local_rank == 0 or args.local_rank == -1:
                checkpoint_dir = f"outputs/checkpoint-{state.global_step}"
                print(f"\n[Rank 0] Step {state.global_step}: Saving progress to {checkpoint_dir}...")
                # Save just the LoRA adapters during training to keep it fast
                model.save_pretrained(checkpoint_dir)

def main():
    max_seq_length = 2048 
    dtype = None 
    load_in_4bit = True 

    # Multi-GPU rank assignment mapping
    device_map = {'': torch.cuda.current_device()}

    # Initialize model and tokenizer
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "unsloth/qwen2.5-1.5b-instruct-bnb-4bit",
        max_seq_length = max_seq_length,
        dtype = dtype,
        load_in_4bit = load_in_4bit,
        device_map = device_map,
    )

    # Apply LoRA targets
    model = FastLanguageModel.get_peft_model(
        model,
        r = 16, 
        target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                          "gate_proj", "up_proj", "down_proj"],
        lora_alpha = 16,
        lora_dropout = 0, 
        bias = "none",    
        use_gradient_checkpointing = "unsloth",
        random_state = 3407,
    )

    # Load xLAM dataset
    dataset = load_dataset("Beryex/xlam-function-calling-60k-sharegpt", split="train")

    # Format tokenizer for ChatML schema
    tokenizer = get_chat_template(
        tokenizer,
        chat_template = "chatml",
        mapping = {"role" : "from", "content" : "value", "user" : "human", "assistant" : "gpt"}
    )

    def apply_chat_template(examples):
        conversations = examples["conversations"]
        texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in conversations]
        return {"text": texts}

    dataset = dataset.map(apply_chat_template, batched=True)

    # Configure distributed trainer for 2 complete epochs
    trainer = SFTTrainer(
        model = model,
        tokenizer = tokenizer,
        train_dataset = dataset,
        dataset_text_field = "text",
        max_seq_length = max_seq_length,
        callbacks = [MultiGpuSaveCallback()], # Add custom callback here
        args = TrainingArguments(
            per_device_train_batch_size = 4, 
            gradient_accumulation_steps = 2, 
            warmup_steps = 150,             
            num_train_epochs = 2,           
            learning_rate = 2e-4,
            fp16 = not torch.cuda.is_bf16_supported(),
            bf16 = torch.cuda.is_bf16_supported(),
            logging_steps = 10,
            optim = "adamw_8bit",
            weight_decay = 0.01,
            lr_scheduler_type = "cosine",
            seed = 3407,
            output_dir = "outputs",
            ddp_find_unused_parameters = False,
            save_strategy = "no", # CRITICAL: Stop the native trainer from breaking process serialization
        ),
    )

    trainer.train()

    # Safely export final merged model state
    if trainer.is_world_process_zero():
        print("\n[Rank 0] Training complete. Saving finalized LoRA adapters...")
        model.save_pretrained_merged("qwen2.5-xlam-lora", tokenizer, save_method="lora")
        print("[Rank 0] Adapters saved successfully into 'qwen2.5-xlam-lora' directory.")

if __name__ == "__main__":
    main()

In [ ]:
!accelerate launch --num_processes 2 train.py

## Evaluation using the NousResearch/json-mode-eval dataset

In [ ]:
import json
import torch
from datasets import load_dataset
from unsloth import FastLanguageModel
from tqdm import tqdm
import os
import shutil
from huggingface_hub import hf_hub_download

def main():
    # 1. Load the fine-tuned LoRA model
    max_seq_length = 2048
    dtype = None
    load_in_4bit = True

    local_model_dir = "/kaggle/working/qwen2.5-xlam-lora"

    # FIX: Fetch missing config.json that was skipped during the 16-bit merge
    config_path = os.path.join(local_model_dir, "config.json")
    if not os.path.exists(config_path):
        print("Fixing missing config files...")
        base_config = hf_hub_download(repo_id="Qwen/Qwen2.5-1.5B-Instruct", filename="config.json")
        shutil.copy(base_config, config_path)

    print("Loading merged adapters...")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = local_model_dir, 
        max_seq_length = max_seq_length,
        dtype = dtype,
        load_in_4bit = load_in_4bit,
    )

    # Enable highly optimized inference speeds
    FastLanguageModel.for_inference(model)

    # 2. Load the evaluation dataset
    print("Loading json-mode-eval dataset...")
    dataset = load_dataset("NousResearch/json-mode-eval", split="train")

    eval_metrics = {
        "total_tested": 0,
        "valid_json": 0,
        "schema_match": 0
    }

    print("Beginning evaluation...")
    
    # 3. Run the evaluation loop
    for sample in tqdm(dataset):
        eval_metrics["total_tested"] += 1
        
        # Format the prompt using ChatML so the model recognizes it as an instruction
        raw_prompt = sample['prompt']
        
        # Handle cases where the dataset prompt is already a list of message dicts
        if isinstance(raw_prompt, list):
            messages = raw_prompt
            # Inject our strict system prompt if one isn't already present
            if len(messages) > 0 and messages[0].get("role") != "system":
                messages.insert(0, {"role": "system", "content": "You are a highly precise data extraction assistant. You must always output raw, valid JSON."})
        else:
            # Handle cases where it is just a plain string
            messages = [
                {"role": "system", "content": "You are a highly precise data extraction assistant. You must always output raw, valid JSON."},
                {"role": "user", "content": str(raw_prompt)}
            ]
        
        # Apply the native ChatML template
        inputs = tokenizer.apply_chat_template(
            messages,
            tokenize = True,
            add_generation_prompt = True,
            return_tensors = "pt",
            return_dict = True,
        ).to("cuda")
        
        # Generate the response (low temperature for strict formatting)
        outputs = model.generate(
            **inputs, 
            max_new_tokens = 512, 
            temperature = 0.1,
            pad_token_id = tokenizer.eos_token_id
        )
        
        # Slice the output to extract only the newly generated tokens
        input_length = inputs["input_ids"].shape[1]
        response = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True).strip()
        
        # 4. Clean up Markdown Code Blocks safely using Hex Escape Sequences
        # \x60 represents a backtick character, preventing parser cutoff errors.
        ticks_json = "\x60\x60\x60json"
        ticks_plain = "\x60\x60\x60"
        
        if response.startswith(ticks_json):
            response = response.replace(ticks_json, "").replace(ticks_plain, "").strip()
        elif response.startswith(ticks_plain):
            response = response.replace(ticks_plain, "").strip()
        
        # 5. Deterministic Scoring
        try:
            # Check 1: Is it structurally valid JSON?
            parsed_output = json.loads(response)
            eval_metrics["valid_json"] += 1
            
            # Check 2: Do the root keys match the ground truth schema?
            ground_truth = json.loads(sample['completion'])
            
            # Type-safe schema matching to avoid AttributeError on JSON arrays
            if type(parsed_output) == type(ground_truth):
                if isinstance(ground_truth, dict):
                    if set(parsed_output.keys()) == set(ground_truth.keys()):
                        eval_metrics["schema_match"] += 1
                elif isinstance(ground_truth, list):
                    # If it's a list of objects, check the keys of the first object
                    if len(parsed_output) > 0 and len(ground_truth) > 0 and isinstance(parsed_output[0], dict) and isinstance(ground_truth[0], dict):
                        if set(parsed_output[0].keys()) == set(ground_truth[0].keys()):
                            eval_metrics["schema_match"] += 1
                    else:
                        # Otherwise just verifying they are both arrays is sufficient
                        eval_metrics["schema_match"] += 1
                else:
                    eval_metrics["schema_match"] += 1
                
        except json.JSONDecodeError:
            # The model hallucinated extra text or broke the JSON syntax
            pass 

    # 6. Calculate and display the final performance
    print("\n" + "="*30)
    print("     FINAL EVALUATION RESULTS     ")
    print("="*30)
    print(f"Total Examples Tested : {eval_metrics['total_tested']}")
    print(f"Valid JSON Formatting : {(eval_metrics['valid_json'] / eval_metrics['total_tested']) * 100:.2f}%")
    print(f"Exact Schema Match    : {(eval_metrics['schema_match'] / eval_metrics['total_tested']) * 100:.2f}%")
    print("="*30)

if __name__ == "__main__":
    main()

## **Base Model test**

In [ ]:
import json
import torch
from datasets import load_dataset
from unsloth import FastLanguageModel
from tqdm import tqdm

def main():
    # 1. Load the RAW BASE MODEL (no adapters)
    max_seq_length = 2048
    dtype = None
    load_in_4bit = True

    print("Loading raw base model...")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "unsloth/qwen2.5-1.5b-instruct-bnb-4bit", 
        max_seq_length = max_seq_length,
        dtype = dtype,
        load_in_4bit = load_in_4bit,
    )

    # Enable highly optimized inference speeds
    FastLanguageModel.for_inference(model)

    # 2. Load the evaluation dataset
    print("Loading json-mode-eval dataset...")
    dataset = load_dataset("NousResearch/json-mode-eval", split="train")

    eval_metrics = {
        "total_tested": 0,
        "valid_json": 0
    }

    print("Beginning evaluation...")
    
    # 3. Run the evaluation loop
    for sample in tqdm(dataset):
        eval_metrics["total_tested"] += 1
        
        # Format the prompt using ChatML so the model recognizes it as an instruction
        raw_prompt = sample['prompt']
        
        # Handle cases where the dataset prompt is already a list of message dicts
        if isinstance(raw_prompt, list):
            messages = raw_prompt
            # Inject our strict system prompt if one isn't already present
            if len(messages) > 0 and messages[0].get("role") != "system":
                messages.insert(0, {"role": "system", "content": "You are a highly precise data extraction assistant. You must always output raw, valid JSON."})
        else:
            # Handle cases where it is just a plain string
            messages = [
                {"role": "system", "content": "You are a highly precise data extraction assistant. You must always output raw, valid JSON."},
                {"role": "user", "content": str(raw_prompt)}
            ]
        
        # Apply the native ChatML template
        inputs = tokenizer.apply_chat_template(
            messages,
            tokenize = True,
            add_generation_prompt = True,
            return_tensors = "pt",
            return_dict = True,
        ).to("cuda")
        
        # Generate the response (low temperature for strict formatting)
        outputs = model.generate(
            **inputs, 
            max_new_tokens = 512, 
            temperature = 0.1,
            pad_token_id = tokenizer.eos_token_id
        )
        
        # Slice the output to extract only the newly generated tokens
        input_length = inputs["input_ids"].shape[1]
        response = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True).strip()
        
        # 4. Clean up Markdown Code Blocks safely using Hex Escape Sequences
        ticks_json = "\x60\x60\x60json"
        ticks_plain = "\x60\x60\x60"
        
        if response.startswith(ticks_json):
            response = response.replace(ticks_json, "").replace(ticks_plain, "").strip()
        elif response.startswith(ticks_plain):
            response = response.replace(ticks_plain, "").strip()
        
        # 5. Deterministic Scoring (JSON Validity Only)
        try:
            # Check 1: Is it structurally valid JSON?
            json.loads(response)
            eval_metrics["valid_json"] += 1
                
        except json.JSONDecodeError:
            # The model hallucinated extra text or broke the JSON syntax
            pass 

    # 6. Calculate and display the final performance
    print("\n" + "="*30)
    print("     FINAL EVALUATION RESULTS     ")
    print("="*30)
    print(f"Total Examples Tested : {eval_metrics['total_tested']}")
    print(f"Valid JSON Formatting : {(eval_metrics['valid_json'] / eval_metrics['total_tested']) * 100:.2f}%")
    print("="*30)

if __name__ == "__main__":
    main()

## Evaluation using the xLAM dataset


In [ ]:
import json
import torch
from datasets import load_dataset
from unsloth import FastLanguageModel
from tqdm import tqdm

def main():
    max_seq_length = 2048
    dtype = None
    load_in_4bit = True

    print("Loading merged adapters...")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name="qwen2.5-xlam-lora", 
        max_seq_length=max_seq_length,
        dtype=dtype,
        load_in_4bit=load_in_4bit,
    )
    FastLanguageModel.for_inference(model)

    print("Loading 100 rows from xLAM dataset...")
    dataset = load_dataset("Beryex/xlam-function-calling-60k-sharegpt", split="train[-100:]")

    eval_metrics = {
        "total_tested": 0,
        "valid_json": 0,
        "exact_match": 0
    }

    print("Beginning evaluation...")

    # 1. Visual Verification Loop
    for i in range(3):
        sample = dataset[i]
        conversations = sample["conversations"]
        
        # BULLETPROOF EXTRACT: Last message is always the answer
        ground_truth = conversations[-1]["value"]
        
        # Everything before the last message forms the prompt
        messages = []
        for turn in conversations[:-1]:
            role = "system" if turn["from"].lower() == "system" else "user"
            messages.append({"role": role, "content": turn["value"]})
            
        inputs = tokenizer.apply_chat_template(
            messages, tokenize=True, add_generation_prompt=True, return_tensors="pt", return_dict=True
        ).to("cuda")
        
        outputs = model.generate(**inputs, max_new_tokens=256, temperature=0.1, pad_token_id=tokenizer.eos_token_id)
        input_length = inputs["input_ids"].shape[1]
        response = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True).strip()
        
        print(f"\n--- VISUAL EXAMPLE {i+1} ---")
        print(f"Ground Truth : {ground_truth}")
        print(f"Model Output : {response}")

    print("\nRunning full validation score across 100 rows...")
    
    # 2. Complete Evaluation Loop
    for sample in tqdm(dataset):
        eval_metrics["total_tested"] += 1
        conversations = sample["conversations"]
        
        # BULLETPROOF EXTRACT
        ground_truth = conversations[-1]["value"]
        
        messages = []
        for turn in conversations[:-1]:
            role = "system" if turn["from"].lower() == "system" else "user"
            messages.append({"role": role, "content": turn["value"]})
            
        inputs = tokenizer.apply_chat_template(
            messages, tokenize=True, add_generation_prompt=True, return_tensors="pt", return_dict=True
        ).to("cuda")
        
        outputs = model.generate(**inputs, max_new_tokens=256, temperature=0.1, pad_token_id=tokenizer.eos_token_id)
        input_length = inputs["input_ids"].shape[1]
        response = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True).strip()
        
        # Clean markdown safely
        ticks_json = "\x60\x60\x60json"
        ticks_plain = "\x60\x60\x60"
        if response.startswith(ticks_json):
            response = response.replace(ticks_json, "").replace(ticks_plain, "").strip()
        elif response.startswith(ticks_plain):
            response = response.replace(ticks_plain, "").strip()
            
        try:
            parsed_out = json.loads(response)
            eval_metrics["valid_json"] += 1
            
            # Safely parse ground truth
            try:
                parsed_truth = json.loads(ground_truth)
                if parsed_out == parsed_truth:
                    eval_metrics["exact_match"] += 1
            except json.JSONDecodeError:
                # If ground truth somehow isn't valid JSON, skip exact match check
                pass
                
        except json.JSONDecodeError:
            pass

    print("\n" + "="*30)
    print("     xLAM EVALUATION RESULTS     ")
    print("="*30)
    print(f"Total Examples Tested : {eval_metrics['total_tested']}")
    print(f"Valid JSON Formatting : {(eval_metrics['valid_json'] / eval_metrics['total_tested']) * 100:.2f}%")
    print(f"Exact Content Match   : {(eval_metrics['exact_match'] / eval_metrics['total_tested']) * 100:.2f}%")
    print("="*30)

if __name__ == "__main__":
    main()